In [ ]:
from kortex_api.autogen.client_stubs.SessionClientRpc import SessionClient
from kortex_api.autogen.client_stubs.BaseClientRpc import BaseClient

from kortex_api.RouterClient import RouterClient
from kortex_api.UDPTransport import UDPTransport
from kortex_api.SessionManager import SessionManager

from kortex_api.autogen.messages import Session_pb2, Base_pb2, ProgramRunner_pb2, Common_pb2, PluginManager_pb2, Plugin_pb2
from kortex_api.autogen.client_stubs.PluginClientRpc import PluginClient
from kortex_api.autogen.client_stubs.PluginManagerClientRpc import PluginManagerClient
from kortex_api.MqttTransport import MqttTransport

from kortex_api.autogen.client_stubs.BaseCyclicClientRpc import BaseCyclicClient
from kortex_api.autogen.messages.Common_pb2 import ModeSelection,OperatingModeType, CartesianReferenceFrame
from kortex_api.exceptions.KServerException import KServerException

from jsonschema import validate
import json

import time
import numpy as np
from google.protobuf import json_format

In [ ]:
import grpc
import armMessages_pb2_grpc
import armMessages_pb2

In [ ]:
errorCallback = lambda kException: print("____CallBack Error____: {}".format(kException))

In [ ]:
CONTROLLER_ADDRESS = "169.254.62.10"
MQTT_PORT = 1883
UDP_PORT = 10001

In [ ]:
homeCart = [0, 200, 480, 0, -90, 90]
homeAngular = [-71.65515899658203,
-50.30020523071289,
-118.6649169921875,
-41.86516571044922,
118.029541015625,
-51.11912536621094]

In [ ]:
def print_all_plugin_actions(plugin):

    print("Available actions in " + plugin.handle.identifier + ":")
    actions:Plugin_pb2.ActionDescription = plugin.actions
    for action in actions:
        print(action.handle)

    print("\n")

In [ ]:
waypoint_count = 1

In [ ]:
class KinovaArm:
    CONTROLLER_ADDRESS = "169.254.62.10"
    MQTT_PORT = 1883
    UDP_PORT = 10001
    connected = False
    armOn = False
    maxLoop = 30
    homeCart = [0, 200, 480, 0, -90, 90]
    homeAngular = [-72.9454651,
                    -50.2982483,
                    -118.821823,
                    -41.3727722,
                    117.561745,
                    -141.523972]
    def __init__(self):
        return
    
    def connectToBase(self):
        try:
            session_info = Session_pb2.CreateSessionInfo()
            session_info.username = "admin"
            session_info.password = "admin"
            session_info.session_inactivity_timeout = 60000 # (milliseconds)
            session_info.connection_inactivity_timeout = 2000 # (milliseconds)

            self.transport = MqttTransport()
            errorCallback = lambda kException: print("____CallBack Error____: {}".format(kException))
            
            router = RouterClient(self.transport, errorCallback)
            self.transport.connect(self.CONTROLLER_ADDRESS, self.MQTT_PORT)

            print("Creating Session")
            self.session_client = SessionClient(router)
            self.session_client.CreateSession(session_info)
            print("Session Created")

            self.base = BaseClient(router)
            print("Connected to the Base")
            
            plugin_name = "robotiq_plugin"
            
            plugin_manager = PluginManagerClient(router)
            plugin_list = plugin_manager.GetPluginsList()
            for plugin in plugin_list.plugin_info_list:
                if plugin.handle.identifier == plugin_name:
                    print(plugin.handle.identifier)
                    print(plugin.plugin_state.state)
                    #print_all_plugin_actions(plugin)
            
            self.gripper_plugin = PluginClient(router, plugin_name)
            
            self.connected = True
            return (1, None)
            
        except Exception as e:
            print("Failed to connect to base; Error: ", e)
            self.connected = False
            return (-1, e)

    def disconnectBase(self):
        try:
            self.session_client.CloseSession()
            self.transport.disconnect()
            return (1, None)
        except Exception as e:
            return (-2, e)
            
        
    def checkArmConnection(self):
        try:
            self.base.GetArmState()
            return True
        except:
            return False
    
    def checkArmPowered(self):
        return(self.base.GetArmState().active_state != Base_pb2.ARMSTATE_IDLE)
        
    def powerArmOn(self):
        try:
            self.base.ActivateRobot()
            return(1, None)
        except Exception as e:
            return(-10, e)
                
    def powerArmOff(self):
        try:
            self.base.DeactivateRobot()
            return(1,None)
        except Exception as e:
            return (-11, e)
    
    def waitForPowerOn(self, timeout=45):
        secs = 0
        state = self.base.GetArmState()
        while state.active_state != Base_pb2.ARMSTATE_ARM_OPERATIONAL:
            if secs > timeout:
                yield (-12, "Failed to turn arm on within the timeout window (it may still yet turn on if timeout was short)")
                return
            print(state)
            if(stat.active_state == Base_pb2.ARMSTATE_INITIALIZATION):
                yield (30 + secs, None)
            elif(state.active_state == Base_pb2.ARMSTATE_BRAKE_RELEASING):
                yield(90)
            
            time.sleep(1)
            state = self.base.GetArmState()
        yield (100, None)
        return
    
    def addNotificationCallback(self, func):
        def logicWrapper(data):
            return
        self.base.OnNotificationActionTopic(logicWrapper, Base_pb2.NotificationOptions())
        
        return
    def change_operating_mode(self, operating_mode_type : str):
        # OPERATING_MODE_UNSPECIFIED (0):       Unspecified operating mode
        # OPERATING_MODE_JOG_MANUAL (1):        Jog manual operating mode
        # OPERATING_MODE_HAND_GUIDING (2):      Hand guiding operating mode
        # OPERATING_MODE_HOLD_TO_RUN (3):       Hold to run operating mode
        # OPERATING_MODE_AUTO (4):              Automatic operating mode
        # OPERATING_MODE_MONITORED_STOP (5):    Monitored stop operating mode

        mode = ModeSelection()
        mode.operating_mode = OperatingModeType.Value(operating_mode_type)
        self.base.SelectOperatingMode(mode)
        return
    
    def stop(self):
        try:
            self.change_operating_mode("OPERATING_MODE_MONITORED_STOP")
            return (1, None)
        except Exception as e:
            
            return (-3, None)
    
    def activateGripper(self):
        activate_input = {}
        action_name = "Activate"
        return self.pluginCommand(action_name, activate_input)
    
    def moveGripper(self, width, velocity=64, force=20):
        maxWidth = 50
        minWidth = 0
        maxVal = 255 #255 is actually fully closed but the math is easier this way
        adjustedVal = int(width * maxVal / (maxWidth - minWidth))
        
        widthVal = 255 - adjustedVal
        
        move_input = {
            "position":width,
            "velocity":velocity,
            "force":force
        }
        action_name = "Move"
        return self.pluginCommand(action_name, move_input)
        
    def pluginCommand(self, action_name, plugin_input):
        try:
            action_list = self.gripper_plugin.GetActionTypes()

            for available_action in action_list.actions:
                if available_action.friendly_name == action_name:
                    action = Plugin_pb2.Action()
                    action.serialization_type= Plugin_pb2.DataType.CONFIGURATION_TYPE_JSON
                    action.handle.CopyFrom(available_action.handle)
                    validate(plugin_input, json.loads(available_action.input_schema))
                    action.input = json.dumps(plugin_input)
                    self.gripper_plugin.StartAction(action)
                    return (1, None)
            return (-4, "Unable to find gripper action, but able to see list of actions")
        except Exception as e:
            return (-5, e)
        
    def moveHome(self):
        return self.angularMoveArm(*self.homeAngular)
    
    def cartesianMoveArm(self, X, Y, Z, xo, yo, zo, dur=0):
        try:
            R = 0
            self.change_operating_mode("OPERATING_MODE_AUTO")

            waypointB = Base_pb2.CartesianWaypoint()
            waypointB.pose.x = X            # in meters
            waypointB.pose.y = Y            # in meters
            waypointB.pose.z = Z            # in meters
            waypointB.blending_radius = R   # in meters
            waypointB.pose.theta_x = xo      # in degrees
            waypointB.pose.theta_y = yo      # in degrees
            waypointB.pose.theta_z = zo      # in degrees

            waypointB.reference_frame = CartesianReferenceFrame.Value("CARTESIAN_REFERENCE_FRAME_BASE")
            #Why the fuck is there all this redundancy here?

            waypointsDefinition = ((X, Y, Z, R, xo, yo, zo))
            waypointCount = len(waypointsDefinition)

            wptlist = Base_pb2.WaypointList()
            wptlist.use_optimal_blending = True


            array_wpts = np.array([])
            index = 0
            for i in range(0,waypointCount):

                np.append(array_wpts, waypointsDefinition[i])
                waypoint = wptlist.waypoints.add()
                waypoint.name = "waypoint_" + str(index)
                waypoint.cartesian_waypoint.CopyFrom(waypointB)
                index = index + 1

            #Add waypoints to waypoint list
            wptlist.waypoints.MergeFrom(array_wpts)
            wptlist.duration = dur # in seconds
            result = self.base.ValidateWaypointList(wptlist)

            if len(result.trajectory_error_report.trajectory_error_elements) == 0:
                print("Reaching cartesian pose trajectory...")
                self.base.ExecuteWaypointTrajectory(wptlist)

            else:
                print("Error found in trajectory")
                print(result.trajectory_error_report)
                return (-6, result.trajectory_error_report)
            return (1, None)
        except Exception as e:
            return (-7, e)
            
        
    def angularMoveArm(self, j1, j2, j3, j4, j5, j6):
        try:
            self.change_operating_mode("OPERATING_MODE_AUTO")

            waypointB = Base_pb2.AngularWaypoint()
            waypointB.angles.MergeFrom([j1, j2, j3, j4, j5, j6])
            waypointB.duration = 0
            waypointB.blending = 0

            waypointsDefinition = (([j1, j2, j3, j4, j5, j6], 0, 1))
            waypointCount = len(waypointsDefinition)

            wptlist = Base_pb2.WaypointList()
            wptlist.use_optimal_blending = True

            # Here we create waypoints from the waypointsDefinition array with a for loop
            array_wpts = np.array([])
            index = 0
            for i in range(0,waypoint_count):

                np.append(array_wpts, waypointsDefinition[i])
                waypoint = wptlist.waypoints.add()
                waypoint.name = "waypoint_" + str(index)
                waypoint.angular_waypoint.CopyFrom(waypointB)
                index = index + 1

            # Add waypoints to waypoint list
            wptlist.waypoints.MergeFrom(array_wpts)
            wptlist.duration = 0 # in seconds
            result = self.base.ValidateWaypointList(wptlist)

            if len(result.trajectory_error_report.trajectory_error_elements) == 0:
                print("Reaching cartesian pose trajectory...")
                self.base.ExecuteWaypointTrajectory(wptlist)
            else:
                print("Error found in trajectory")
                print(result.trajectory_error_report)
                return (-8, result.trajectory_error_report)
        except Exception as e:
            return (-9, e)
    
    def inverseKinematics(self, X, Y, Z, tX, tY, tZ, guess=[-91.66485595703125, 3.00946307182312, -99.51344299316406,
                -10.15340805053711, 77.40167999267578, -79.85208892822266]):
        try:
            input_joint_angles = self.base.GetMeasuredJointAngles()
            pose = self.base.GetMeasuredCartesianPose()
        except KServerException as ex:
            print("Unable to get current robot pose")
            print(
                "Error_code:{} , Sub_error_code:{} ".format(
                    ex.get_error_code(), ex.get_error_sub_code()
                )
            )
            print("Caught expected error: {}".format(ex))
            return None
        
        input_IkData = Base_pb2.IKData()
        # Fill the IKData Object with the cartesian coordinates that need to be converted
        input_IkData.cartesian_pose.x = X
        input_IkData.cartesian_pose.y = Y
        input_IkData.cartesian_pose.z = Z
        input_IkData.cartesian_pose.theta_x = tX
        input_IkData.cartesian_pose.theta_y = tY
        input_IkData.cartesian_pose.theta_z = tZ

        # Fill the IKData Object with the guessed joint angles
        for i, joint_angle in enumerate(input_joint_angles.joint_angles):
            jAngle = input_IkData.guess.joint_angles.add()
            # '- 1' to generate an actual "guess" for current joint angles
            jAngle.value = joint_angle.value - 10
        try:
            print("Computing Inverse Kinematics using joint angles and pose...")
            computed_joint_angles = self.base.ComputeInverseKinematics(input_IkData)
        except KServerException as ex:
            print("Unable to compute inverse kinematics")
            print(
                "Error_code:{} , Sub_error_code:{} ".format(
                    ex.get_error_code(), ex.get_error_sub_code()
                )
            )
            print("Caught expected error: {}".format(ex))
            return None

        print("Joint ID : Joint Angle")
        joint_identifier = 0
        for joint_angle in computed_joint_angles.joint_angles:
            print(joint_identifier, " : ", joint_angle.value)
            joint_identifier += 1

        return computed_joint_angles

In [ ]:
def notification_callback1(data):
    data = json_format.MessageToDict(data)
    try:
        if data["actionEvent"] == "ACTION_FEEDBACK":
            return
    except:
        print("FUCK")
        print(dir(data))
        return
    print("****************************")
    print("* Callback function1 called *")
    print(data)
    print("****************************")


In [86]:
def notification_callback2(data):
    data = json_format.MessageToDict(data)
    print("****************************")
    print("* Callback function2 called *")
    print(data)
    print("****************************")

In [88]:
x = KinovaArm()

In [89]:
x.connectToBase()
print(x.base.GetArmState())
print(x.base.GetMeasuredCartesianPose())

Creating Session
Session Created
Connected to the Base
robotiq_plugin
4
active_state: ARMSTATE_ARM_OPERATIONAL

x: 0.0240460448
y: -0.17331101
z: 0.458524317
theta_x: 88.6722336
theta_y: -179.777695
theta_z: 2.9244957



In [90]:
feedbackHandle1 = x.base.OnNotificationActionTopic(notification_callback1, Base_pb2.NotificationOptions())

In [91]:
feedbackHandle2 = x.gripper_plugin.OnNotificationActionTopic(notification_callback2, Plugin_pb2.NotificationOptions())

In [104]:
x.powerArmOn()

(1, None)

In [ ]:
#print(dir(Base_pb2))
for i in dir(Base_pb2):
    print(i)

In [114]:
x.activateGripper()

(1, None)

****************************
* Callback function2 called *
{'actionEvent': 'ACTION_START', 'instanceHandle': {'identifier': 3}, 'handle': {}, 'genericInfo': {}}
****************************
****************************
* Callback function2 called *
{'actionEvent': 'ACTION_END', 'instanceHandle': {'identifier': 3}, 'applicationData': '{"result":"SUCCESS"}', 'handle': {}, 'genericInfo': {}}
****************************


In [93]:
x.moveGripper(231, 16, 1)

(1, None)

****************************
* Callback function2 called *
{'actionEvent': 'ACTION_START', 'instanceHandle': {'identifier': 2}, 'handle': {}, 'genericInfo': {}}
****************************
****************************
* Callback function2 called *
{'actionEvent': 'ACTION_END', 'instanceHandle': {'identifier': 2}, 'applicationData': '{"fault":0,"object_detected":false,"position":231.0,"requested_position":231.0}', 'handle': {}, 'genericInfo': {}}
****************************


In [ ]:
x.powerArmOn()
#for i in x.waitForPowerOn():
#    print(i)

In [106]:
print(x.base.GetArmState())
print(x.base.GetArmState().active_state)
print(x.base.GetArmState() == )

active_state: ARMSTATE_IN_FAULT

4


In [116]:
print(x.gripper_plugin.GetStatus())


state: STATE_ACTIVE
details: "Failed to initialize communication with the wrist."



In [103]:
for i in dir(x.base):
    if "Err" in i:
        print(i)

GetTrajectoryErrorReport


In [ ]:
print(x.base.GetMeasuredCartesianPose())
print(x.base.GetMeasuredJointAngles())

In [ ]:
x.cartesianMoveArm(0, -.2, .480, 90, -180, 0)

In [ ]:
x.cartesianMoveArm(0.3, -0.400, .48, 0, -90.0, 90.0)

In [ ]:
x.angularMoveArm(-127.05558013916016,
1.4330555200576782,
-68.37745666503906,
-0.2931045889854431,
200.6851348876953,
-126.68273162841797)

In [105]:
x.cartesianMoveArm(0, -0.6, 0.43, 90, -180, 90)

Reaching cartesian pose trajectory...


(1, None)

****************************
* Callback function1 called *
{'actionEvent': 'ACTION_PREPROCESS_END', 'handle': {'actionType': 'EXECUTE_WAYPOINT_LIST'}, 'genericInfo': {'timestamp': {'sec': 1689973482, 'usec': 187469}, 'connection': {'userHandle': {'identifier': 1}, 'connectionInformation': 'res/auto-23D0A3B0-75BA-5471-905D-AF0D528C8A4B'}}}
****************************
****************************
* Callback function1 called *
{'actionEvent': 'ACTION_START', 'handle': {'actionType': 'EXECUTE_WAYPOINT_LIST'}, 'genericInfo': {'timestamp': {'sec': 1689973482, 'usec': 188583}, 'connection': {'userHandle': {'identifier': 1}, 'connectionInformation': 'res/auto-23D0A3B0-75BA-5471-905D-AF0D528C8A4B'}}}
****************************
****************************
* Callback function1 called *
{'actionEvent': 'ACTION_ABORT', 'handle': {'actionType': 'EXECUTE_WAYPOINT_LIST'}, 'genericInfo': {'timestamp': {'sec': 1689973486, 'usec': 94390}, 'connection': {'userHandle': {'identifier': 1}, 'connectionInf

In [109]:
x.stop()

(1, None)

****************************
* Callback function1 called *
{'actionEvent': 'ACTION_END', 'handle': {'actionType': 'EXECUTE_WAYPOINT_LIST'}, 'genericInfo': {'timestamp': {'sec': 1689973770, 'usec': 561479}, 'connection': {'userHandle': {'identifier': 1}, 'connectionInformation': 'res/auto-23D0A3B0-75BA-5471-905D-AF0D528C8A4B'}}}
****************************


In [107]:
x.base.ClearFaults()
time.sleep(.3)
x.moveHome()

Reaching cartesian pose trajectory...
****************************
* Callback function1 called *
{'actionEvent': 'ACTION_PREPROCESS_END', 'handle': {'actionType': 'EXECUTE_WAYPOINT_LIST'}, 'genericInfo': {'timestamp': {'sec': 1689973558, 'usec': 771467}, 'connection': {'userHandle': {'identifier': 1}, 'connectionInformation': 'res/auto-23D0A3B0-75BA-5471-905D-AF0D528C8A4B'}}}
****************************
****************************
* Callback function1 called *
{'actionEvent': 'ACTION_START', 'handle': {'actionType': 'EXECUTE_WAYPOINT_LIST'}, 'genericInfo': {'timestamp': {'sec': 1689973558, 'usec': 772481}, 'connection': {'userHandle': {'identifier': 1}, 'connectionInformation': 'res/auto-23D0A3B0-75BA-5471-905D-AF0D528C8A4B'}}}
****************************
****************************
* Callback function1 called *
{'actionEvent': 'ACTION_END', 'handle': {'actionType': 'EXECUTE_WAYPOINT_LIST'}, 'genericInfo': {'timestamp': {'sec': 1689973562, 'usec': 648474}, 'connection': {'userHandl

In [ ]:
x.base.Unsubscribe(feedbackHandle)

In [ ]:
jangles = x.inverseKinematics(.100, -.400, .480, 0, -90, 90)
print(list(i.value for i in jangles.joint_angles)) 

In [ ]:
import itertools
X = [-.100, 0, .100]
Y = [-.6]
Z = [.43, .48, .53]
dX = [0]
dY = [-90]
dZ = [90]
combo = list(itertools.product(X, Y, Z, dX, dY, dZ))
print(combo)

In [ ]:
print(combo[3])
print(combo[5])
print(combo[8])

In [ ]:
jangles = x.inverseKinematics(*combo[4])
prior = list(i.value for i in jangles.joint_angles)
print(prior)

In [ ]:
jangles = x.inverseKinematics(*combo[5], guess=[0,0,0,0,0,0])
print(list(i.value for i in jangles.joint_angles))

In [ ]:
x.angularMoveArm(*homeAngular)

In [ ]:
savedAngles = []

In [ ]:
[-127.05558013916016,
1.4330555200576782,
-68.37745666503906,
-0.2931045889854431,
200.6851348876953,
-126.68273162841797]

In [ ]:
x.inverseKinematics(0, -.200, .480, 0, -90, 90)
#x.inverseKinematics(0, -0.6, 0.48, 0, 180, 90)

In [ ]:
count = 0
for pos in combo:
    print(count)
    print("Moving to: ", pos)
    ik = x.inverseKinematics(*pos)
    jangles = list(i.value for i in ik.joint_angles)
    savedAngles.append((pos, jangles))
    x.angularMoveArm(*jangles)
    print("Moving to Position")
    tmp = input()
    x.angularMoveArm(*homeAngular)
    print("Moving Home")
    tmp = input()
    count += 1

In [ ]:
x.stop()

In [ ]:
action_list = x.gripper_plugin.GetActionTypes()

action_name = "Move"
activate_input = {
            "position":128,
            "velocity":64,
            "force":20
        }
for available_action in action_list.actions:
    if available_action.friendly_name == action_name:
        action = Plugin_pb2.Action()
        action.serialization_type= Plugin_pb2.DataType.CONFIGURATION_TYPE_JSON
        action.handle.CopyFrom(available_action.handle)
        print("Activate Found")
        validate(activate_input, json.loads(available_action.input_schema))
        action.input = json.dumps(activate_input)
        x.gripper_plugin.StartAction(action)

In [ ]:
x.activateGripper()

In [ ]:
x.moveGripper(210)

In [117]:
x.powerArmOff()

(1, None)

In [85]:
x.disconnectBase()

(1, None)

2023-07-19 12:01:44,640 failed to receive on socket: [WinError 10054] An existing connection was forcibly closed by the remote host
2023-07-19 12:01:44,640 failed to receive on socket: [WinError 10054] An existing connection was forcibly closed by the remote host
2023-07-19 12:01:44,641 Unexpected disconnect: The connection was lost.
2023-07-19 12:01:44,642 Unexpected disconnect: The connection was lost.
2023-07-20 16:48:20,468 failed to receive on socket: [WinError 10054] An existing connection was forcibly closed by the remote host
2023-07-20 16:48:20,468 failed to receive on socket: [WinError 10054] An existing connection was forcibly closed by the remote host
2023-07-20 16:48:20,469 Unexpected disconnect: The connection was lost.
2023-07-20 16:48:20,470 Unexpected disconnect: The connection was lost.


In [ ]:
session_info = Session_pb2.CreateSessionInfo()
session_info.username = "admin"
session_info.password = "admin"
session_info.session_inactivity_timeout = 60000 # (milliseconds)
session_info.connection_inactivity_timeout = 2000 # (milliseconds)

transport = UDPTransport()
router = RouterClient(transport, errorCallback)

transport.connect(CONTROLLER_ADDRESS, UDP_PORT)

print("Creating Session")
session_manager = SessionManager(router)
print("Check2")
session_manager.CreateSession(session_info)

print("Session created")
cyclic = BaseCyclicClient(router)
print(cyclic.RefreshFeedback())

session_manager.CloseSession()

transport.disconnect()

In [ ]:
class armCommunicationServicer(armMessages_pb2_grpc.armCommunicationServicer):
    
    def __init__(self, armConnection):
        self.armConnection = armConnection
        return
    
    
    def armStatus(self, request, context):
        
        return armMessages_pb2.statusResponse(flag = 1)

    def armControl(self, request, context):
        
        return
    
    def armFeedback(self, request, context):
        for i in feedbackqueue:
            yield i

In [ ]:
X = 381
Y = 580
Z = 600

In [ ]:
Q = (X * X) / (970 * 970) + (Y * Y) / (970 * 970) + (Z * Z) / (860 * 860)
print(Q)

In [ ]:
import time
t = time.time()
print(t)
print(type(t))
time.sleep(.001)
t2 = time.time()
print(t2 - t)

In [ ]:
class circ:
    def __init__(self):
        self.data = [0] * 6
    def __add__(self, o):
        if(isinstance(o, square)):
            print("adding square")
            r = circ()
            r.data = self.data + o.data
            return r
            
        elif(isinstance(o, tri)):
            print("adding tri")
            r = circ()
            r.data = self.data + o.data
            return r
        
        elif(isinstance(o, circ)):
            print("adding circle")
            r = circ()
            r.data = self.data + o.data
            return r
            
        else:
            print("The fuck is happening")
class square:
    def __init__(self):
        self.data = [1] * 4

class tri:
    def __init__(self):
        self.data = [2] * 3
        
    

In [ ]:
c1 = circ()
c2 = circ()
s1 = square()
s2 = square()
t1 = tri()
t2 = tri()

In [ ]:
print((c1 + s1 + t1).data)
print((c2 + t2 + s2).data)

In [ ]:
class test:
    def __init__(self):
        print("In test init")
        self.x = 1
    def checkRet(self):
        return self

In [ ]:
def mathStuff(width):
    maxWidth = 50
    minWidth = 0
    maxVal = 255 #255 is actually fully closed but the math is easier this way
    minVal = 0
    adjustedVal = int(width * (maxVal - minVal)/ (maxWidth - minWidth))
    print(adjustedVal)

    widthVal = 255 - adjustedVal
    print(widthVal)

In [ ]:
mathStuff(10)

In [ ]:
def ycheck():
    i = 0
    while(True):
        if i > 5:
            yield -1
            return
        i+=1
        yield i

In [ ]:
for i in ycheck():
    print(i)

In [118]:
(.6*.6 + .381 * .381 + .48 * .48)**.5

0.8576485294105038

2023-07-21 16:56:08,932 failed to receive on socket: [WinError 10054] An existing connection was forcibly closed by the remote host
2023-07-21 16:56:08,933 failed to receive on socket: [WinError 10054] An existing connection was forcibly closed by the remote host
2023-07-21 16:56:08,933 failed to receive on socket: [WinError 10054] An existing connection was forcibly closed by the remote host
2023-07-21 16:56:08,934 Unexpected disconnect: The connection was lost.
2023-07-21 16:56:08,934 Unexpected disconnect: The connection was lost.
2023-07-21 16:56:08,935 Unexpected disconnect: The connection was lost.


In [120]:
round(-185/90) * 90

-180

2023-07-25 17:36:31,764 failed to receive on socket: [WinError 10054] An existing connection was forcibly closed by the remote host
2023-07-25 17:36:31,764 failed to receive on socket: [WinError 10054] An existing connection was forcibly closed by the remote host
2023-07-25 17:36:31,764 failed to receive on socket: [WinError 10054] An existing connection was forcibly closed by the remote host
2023-07-25 17:36:31,765 Unexpected disconnect: The connection was lost.
2023-07-25 17:36:31,766 Unexpected disconnect: The connection was lost.
2023-07-25 17:36:31,767 Unexpected disconnect: The connection was lost.
